In [4]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
import warnings

# --------------------------------------------------
# paths
# --------------------------------------------------
NETID = "k16v981"  # change if needed

BASE_DIR = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data"
WBT_DIR = os.path.join(BASE_DIR, "DailyPeakState")
WBT_GLOB = os.path.join(WBT_DIR, "DailyPeakState-*.nc")
IDX_CSV = os.path.join(BASE_DIR, "sst", "roni_dmi_monthly_1950_2025.csv")
OUT_DIR = os.path.join(BASE_DIR, "wbt_sst_city_runs")
os.makedirs(OUT_DIR, exist_ok=True)

CITIES = {
    "muscat": {"lat": 23.5880, "lon": 58.3829},
    "doha":   {"lat": 25.2854, "lon": 51.5310},
    "dubai":  {"lat": 25.2048, "lon": 55.2708},
    "jeddah": {"lat": 21.4858, "lon": 39.1925},
    "aden":   {"lat": 12.7855, "lon": 45.0187},
}

WBT_VAR = "wbt_daily_peak"
MAX_LAG = 6
JJAS_MONTHS = [6, 7, 8, 9]

# --------------------------------------------------
# helpers
# --------------------------------------------------
def to_0360(lon):
    return lon % 360

def infer_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims:
            return c
    return None

def standardize_time_coord(ds):
    # Rename date -> time if needed
    if "time" in ds.coords or "time" in ds.dims:
        return ds, "time"
    if "day" in ds.coords or "day" in ds.dims:
        ds = ds.rename({"day": "time"})
        return ds, "time"
    return ds, None

def open_one_file(path):
    # h5netcdf is often quieter/more forgiving for these files
    # fallback to default if unavailable
    try:
        ds = xr.open_dataset(path, engine="h5netcdf")
    except Exception:
        ds = xr.open_dataset(path)
    return ds

# --------------------------------------------------
# inspect one file first
# --------------------------------------------------
files = sorted(glob.glob(WBT_GLOB))
if not files:
    raise FileNotFoundError(f"No files matched {WBT_GLOB}")

print(f"Found {len(files)} WBT files")
sample = open_one_file(files[0])

print("\n--- SAMPLE FILE ---")
print(sample)
print("\ncoords:", list(sample.coords))
print("dims:", dict(sample.dims))
print("data_vars:", list(sample.data_vars))

if WBT_VAR not in sample.data_vars:
    raise ValueError(f"{WBT_VAR!r} not found in sample file.")

sample, time_name = standardize_time_coord(sample)
lat_name = infer_coord_name(sample, ["lat", "latitude", "y"])
lon_name = infer_coord_name(sample, ["lon", "longitude", "x"])

print("\nDetected:")
print("time_name:", time_name)
print("lat_name :", lat_name)
print("lon_name :", lon_name)

if time_name is None:
    raise ValueError(
        "Could not find a time coordinate. Check whether the files use something other than 'time' or 'date'."
    )
if lat_name is None or lon_name is None:
    raise ValueError(
        "Could not infer lat/lon coordinate names."
    )

# determine longitude convention
use_0360 = float(sample[lon_name].max()) > 180
print("longitude is 0..360:", use_0360)

# --------------------------------------------------
# extract city daily WBT from each file
# --------------------------------------------------
all_rows = []

for i, f in enumerate(files, start=1):
    if i % 25 == 0 or i == 1 or i == len(files):
        print(f"Processing file {i}/{len(files)}: {os.path.basename(f)}")

    try:
        ds = open_one_file(f)
        ds, this_time_name = standardize_time_coord(ds)

        if this_time_name is None:
            print(f"  skipping {os.path.basename(f)}: no time/date coordinate")
            continue

        if WBT_VAR not in ds.data_vars:
            print(f"  skipping {os.path.basename(f)}: {WBT_VAR} missing")
            continue

        # keep only the one variable
        da = ds[WBT_VAR]

        for city, meta in CITIES.items():
            lon = to_0360(meta["lon"]) if use_0360 else meta["lon"]

            point = da.sel(
                {lat_name: meta["lat"], lon_name: lon},
                method="nearest"
            )

            # force to dataframe with a known column name
            s = point.to_series()
            df = s.reset_index()
            value_col = df.columns[-1]
            df = df.rename(columns={value_col: "wbt_daily_peak"})

            # robustly determine time column after reset_index
            possible_time_cols = ["time", "date"]
            actual_time_col = None
            for c in possible_time_cols:
                if c in df.columns:
                    actual_time_col = c
                    break

            if actual_time_col is None:
                # fall back: pick any datetime-like column
                for c in df.columns:
                    if np.issubdtype(df[c].dtype, np.datetime64):
                        actual_time_col = c
                        break

            if actual_time_col is None:
                print(f"  skipping {city} in {os.path.basename(f)}: no time-like column after reset_index")
                continue

            keep_cols = [actual_time_col, "wbt_daily_peak"]
            if lat_name in df.columns:
                keep_cols.append(lat_name)
            if lon_name in df.columns:
                keep_cols.append(lon_name)

            df = df[keep_cols].copy()
            df = df.rename(columns={actual_time_col: "time"})
            df["city"] = city

            all_rows.append(df)

    except Exception as e:
        print(f"  failed on {os.path.basename(f)}: {e}")

if not all_rows:
    raise ValueError("No city rows were extracted from any WBT files.")

city_daily_wbt = pd.concat(all_rows, ignore_index=True)
city_daily_wbt["time"] = pd.to_datetime(city_daily_wbt["time"])
city_daily_wbt = city_daily_wbt.sort_values(["city", "time"]).reset_index(drop=True)

print("\ncity_daily_wbt head:")
print(city_daily_wbt.head())
print(city_daily_wbt.shape)

# --------------------------------------------------
# monthly WBT metrics
# --------------------------------------------------
city_daily_wbt["ym"] = city_daily_wbt["time"].dt.to_period("M").dt.to_timestamp()

city_monthly_wbt = (
    city_daily_wbt
    .groupby(["city", "ym"], as_index=False)
    .agg(
        wbt_mean=("wbt_daily_peak", "mean"),
        wbt_p95=("wbt_daily_peak", lambda x: np.nanpercentile(x, 95)),
        wbt_p99=("wbt_daily_peak", lambda x: np.nanpercentile(x, 99)),
        n_days=("wbt_daily_peak", "count"),
    )
)

city_monthly_wbt["year"] = city_monthly_wbt["ym"].dt.year
city_monthly_wbt["month"] = city_monthly_wbt["ym"].dt.month
city_monthly_wbt_jjas = city_monthly_wbt[
    city_monthly_wbt["month"].isin(JJAS_MONTHS)
].copy()

# --------------------------------------------------
# open RONI / DMI
# --------------------------------------------------
idx = pd.read_csv(IDX_CSV)
print("\nIndex columns:", list(idx.columns))

year_col = next((c for c in idx.columns if c.lower() == "year"), None)
month_col = next((c for c in idx.columns if c.lower() == "month"), None)
roni_col = next((c for c in idx.columns if "roni" in c.lower()), None)
dmi_col = next((c for c in idx.columns if "dmi" in c.lower()), None)

if year_col is not None and month_col is not None:
    idx["ym"] = pd.to_datetime(
        dict(year=idx[year_col].astype(int), month=idx[month_col].astype(int), day=1)
    )
else:
    time_col = next((c for c in idx.columns if c.lower() in ["time", "date", "datetime"]), None)
    if time_col is None:
        raise ValueError("Could not infer date column in RONI/DMI CSV.")
    idx["ym"] = pd.to_datetime(idx[time_col]).dt.to_period("M").dt.to_timestamp()

if roni_col is None or dmi_col is None:
    raise ValueError("Could not infer RONI/DMI columns in the CSV.")

idx_monthly = (
    idx[["ym", roni_col, dmi_col]]
    .rename(columns={roni_col: "RONI", dmi_col: "DMI"})
    .sort_values("ym")
    .drop_duplicates("ym")
    .reset_index(drop=True)
)

idx_lagged = idx_monthly.copy()
for var in ["RONI", "DMI"]:
    for lag in range(MAX_LAG + 1):
        idx_lagged[f"{var}_lag{lag}"] = idx_lagged[var].shift(lag)

# --------------------------------------------------
# merge
# --------------------------------------------------
merged_allmonths = city_monthly_wbt.merge(idx_lagged, on="ym", how="inner")
merged_jjas = city_monthly_wbt_jjas.merge(idx_lagged, on="ym", how="inner")

# --------------------------------------------------
# save
# --------------------------------------------------
city_daily_wbt.to_csv(os.path.join(OUT_DIR, "city_daily_wbt.csv"), index=False)
city_monthly_wbt.to_csv(os.path.join(OUT_DIR, "city_monthly_wbt.csv"), index=False)
city_monthly_wbt_jjas.to_csv(os.path.join(OUT_DIR, "city_monthly_wbt_JJAS.csv"), index=False)
idx_monthly.to_csv(os.path.join(OUT_DIR, "roni_dmi_monthly.csv"), index=False)
idx_lagged.to_csv(os.path.join(OUT_DIR, "roni_dmi_monthly_lagged_0_6.csv"), index=False)
merged_allmonths.to_csv(os.path.join(OUT_DIR, "city_wbt_roni_dmi_lagged_allmonths.csv"), index=False)
merged_jjas.to_csv(os.path.join(OUT_DIR, "city_wbt_roni_dmi_lagged_JJAS.csv"), index=False)

print("\nSaved:")
print("  city_daily_wbt.csv")
print("  city_monthly_wbt.csv")
print("  city_monthly_wbt_JJAS.csv")
print("  roni_dmi_monthly.csv")
print("  roni_dmi_monthly_lagged_0_6.csv")
print("  city_wbt_roni_dmi_lagged_allmonths.csv")
print("  city_wbt_roni_dmi_lagged_JJAS.csv")

Found 900 WBT files

--- SAMPLE FILE ---
<xarray.Dataset> Size: 18MB
Dimensions:                 (day: 31, longitude: 105, latitude: 97)
Coordinates:
  * day                     (day) datetime64[ns] 248B 1950-01-01 ... 1950-01-31
  * longitude               (longitude) float32 420B 34.0 34.25 ... 59.75 60.0
  * latitude                (latitude) float32 388B 34.0 33.75 ... 10.25 10.0
Data variables: (12/14)
    wbt_daily_peak          (day, latitude, longitude) float32 1MB ...
    hour_of_wbt_daily_peak  (day, latitude, longitude) float32 1MB ...
    t2m_daily_peak          (day, latitude, longitude) float32 1MB ...
    hour_of_t2m_daily_peak  (day, latitude, longitude) float32 1MB ...
    t2m_at_wbt_daily_peak   (day, latitude, longitude) float32 1MB ...
    q_daily_peak            (day, latitude, longitude) float32 1MB ...
    ...                      ...
    rh_daily_peak           (day, latitude, longitude) float32 1MB ...
    hour_of_rh_daily_peak   (day, latitude, longitude) floa

/tmp/ipykernel_27975/2124019393.py:75: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("dims:", dict(sample.dims))


Processing file 25/900: DailyPeakState-195201.nc
Processing file 50/900: DailyPeakState-195402.nc
Processing file 75/900: DailyPeakState-195603.nc
Processing file 100/900: DailyPeakState-195804.nc
Processing file 125/900: DailyPeakState-196005.nc
Processing file 150/900: DailyPeakState-196206.nc
Processing file 175/900: DailyPeakState-196407.nc
Processing file 200/900: DailyPeakState-196608.nc
Processing file 225/900: DailyPeakState-196809.nc
Processing file 250/900: DailyPeakState-197010.nc
Processing file 275/900: DailyPeakState-197211.nc
Processing file 300/900: DailyPeakState-197412.nc
Processing file 325/900: DailyPeakState-197701.nc
Processing file 350/900: DailyPeakState-197902.nc
Processing file 375/900: DailyPeakState-198103.nc
Processing file 400/900: DailyPeakState-198304.nc
Processing file 425/900: DailyPeakState-198505.nc
Processing file 450/900: DailyPeakState-198706.nc
Processing file 475/900: DailyPeakState-198907.nc
Processing file 500/900: DailyPeakState-199108.nc
Pro

In [2]:
import pandas as pd


df = pd.read_csv("../data/wbt_sst_city_runs/city_daily_wbt_JJAS_with_lagged_phases.csv")

In [4]:
print(df.columns)

Index(['time', 'wbt_daily_peak', 'city', 'year', 'month', 'ym', 'p95', 'p99',
       'n_days', 'exceed_p95', 'exceed_p99', 'lag', 'RONI_lag', 'DMI_lag',
       'phase'],
      dtype='object')
